# Transfer Learning para Clasificación de Flores (Oxford Flowers 102)
**Objetivo:** comparar *entrenamiento desde cero* vs. *transfer learning* usando una arquitectura **ResNet**.

Dataset: **Oxford Flowers 102** (102 clases). Se descarga automáticamente vía `torchvision.datasets.Flowers102`.

## Grupos a comparar
- **Grupo A:** ResNet entrenada desde cero (**50 épocas**)
- **Grupo B:** Fine-tuning de ResNet preentrenada (**20 épocas**)
- **Grupo C:** Feature extraction (backbone congelado) + nuevo clasificador (**10 épocas**)

## Métricas
- **Accuracy final** (test)
- **Tiempo de entrenamiento** (seg)
- **Épocas hasta convergencia** (criterio: no mejora en *val accuracy* por `patience` épocas)
- **Uso de memoria GPU** (peak, MB)

> Nota: Para que la comparación sea *justa*, se mantienen idénticos: splits, augmentations, optimizador/base LR (con ajustes mínimos), batch size y pipeline de evaluación.


## 0) Setup
Este notebook está diseñado para ejecutarse en GPU (recomendado). En CPU funcionará, pero será lento.

**Requisitos (PyTorch):** `torch`, `torchvision`, `numpy`, `pandas`, `matplotlib`.

Si no tienes GPU, puedes bajar el tamaño del modelo (ResNet18 ya es liviana), reducir `batch_size`, o disminuir épocas.


In [1]:

# Instalación opcional (si estás en Google Colab, normalmente ya está instalado)
# !pip -q install torch torchvision

import os
import time
import math
import copy
import json
import random
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms
from torchvision.datasets import Flowers102
from torchvision.models import resnet18, ResNet18_Weights

# Reproducibilidad (determinista hasta donde es razonable)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True  # rendimiento
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


ModuleNotFoundError: No module named 'torch'

## 1) Dataset y DataLoaders
Usaremos `Flowers102` con sus splits oficiales: **train / val / test**.

### Transformaciones
- **Train:** `RandomResizedCrop`, `RandomHorizontalFlip`, normalización ImageNet
- **Val/Test:** `Resize` + `CenterCrop`, normalización ImageNet

Esto es estándar para transfer learning con modelos preentrenados en ImageNet.


In [ ]:

# Hiperparámetros base
IMG_SIZE = 224
BATCH_SIZE = 64  # ajusta según VRAM
NUM_WORKERS = 2  # en Windows podría requerir 0
PIN_MEMORY = torch.cuda.is_available()

# Normalización ImageNet
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

eval_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# Descarga y carga (requiere internet para descargar el dataset la primera vez)
data_root = "./data_flowers102"

train_ds = Flowers102(root=data_root, split="train", download=True, transform=train_tfms)
val_ds   = Flowers102(root=data_root, split="val",   download=True, transform=eval_tfms)
test_ds  = Flowers102(root=data_root, split="test",  download=True, transform=eval_tfms)

num_classes = 102
len(train_ds), len(val_ds), len(test_ds), num_classes


In [ ]:

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


## 2) Utilidades: entrenamiento, evaluación y medición
Medimos:
- **Tiempo** con `time.perf_counter()`
- **Memoria GPU (peak)** con `torch.cuda.max_memory_allocated()`
- **Convergencia** usando *early stopping* basado en `val_accuracy` (sin detener necesariamente el experimento, pero registrando la primera época donde se alcanza el mejor score y deja de mejorar por `patience`).

Usamos **AMP (mixed precision)** cuando hay GPU para mejorar velocidad y uso de memoria.


In [ ]:

@dataclass
class TrainConfig:
    group_name: str
    epochs: int
    lr: float
    weight_decay: float = 1e-4
    patience: int = 5  # para "épocas hasta convergencia"
    use_amp: bool = True

def accuracy_top1(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    total_n = 0

    ce = nn.CrossEntropyLoss()
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = ce(logits, y)

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_acc  += accuracy_top1(logits, y) * bs
        total_n += bs

    return {"loss": total_loss / total_n, "acc": total_acc / total_n}

def reset_peak_gpu_mem():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

def get_peak_gpu_mem_mb() -> float:
    if not torch.cuda.is_available():
        return float('nan')
    return torch.cuda.max_memory_allocated() / (1024**2)

def train_one_experiment(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    cfg: TrainConfig,
) -> Dict[str, object]:
    model = model.to(device)
    ce = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    # Cosine schedule (simple y robusto)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and torch.cuda.is_available()))

    history = {"epoch": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}

    best_state = None
    best_val_acc = -1.0
    best_epoch = -1
    no_improve = 0
    convergence_epoch = None  # primera época donde se identifica estancamiento tras el mejor

    reset_peak_gpu_mem()
    t0 = time.perf_counter()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        running_loss = 0.0
        running_acc = 0.0
        n = 0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(cfg.use_amp and torch.cuda.is_available())):
                logits = model(x)
                loss = ce(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            bs = x.size(0)
            running_loss += loss.item() * bs
            running_acc  += accuracy_top1(logits, y) * bs
            n += bs

        scheduler.step()

        train_loss = running_loss / n
        train_acc  = running_acc / n
        val_metrics = evaluate(model, val_loader)

        current_lr = optimizer.param_groups[0]["lr"]

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["acc"])
        history["lr"].append(current_lr)

        # Early-stopping style tracking (solo para medir convergencia)
        if val_metrics["acc"] > best_val_acc + 1e-4:
            best_val_acc = val_metrics["acc"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if convergence_epoch is None and no_improve >= cfg.patience:
                convergence_epoch = epoch  # se detecta estancamiento

        if epoch % 1 == 0:
            print(f"[{cfg.group_name}] Epoch {epoch:03d}/{cfg.epochs} | "
                  f"train_acc={train_acc:.3f} val_acc={val_metrics['acc']:.3f} | lr={current_lr:.2e}")

    total_time = time.perf_counter() - t0
    peak_mem_mb = get_peak_gpu_mem_mb()

    # Restaurar mejor modelo según validación
    if best_state is not None:
        model.load_state_dict(best_state)

    result = {
        "group": cfg.group_name,
        "history": pd.DataFrame(history),
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
        "convergence_epoch": convergence_epoch if convergence_epoch is not None else cfg.epochs,
        "train_time_sec": total_time,
        "peak_gpu_mem_mb": peak_mem_mb,
        "model": model,
    }
    return result

@torch.no_grad()
def test_model(model: nn.Module, loader: DataLoader) -> Dict[str, float]:
    return evaluate(model, loader)


## 3) Definición de modelos por grupo
Para la comparación usamos **ResNet18** (rápida y estándar). Puedes cambiar a ResNet50 si tienes más GPU.

- **Grupo A (from scratch):** `weights=None`, se inicializa aleatoriamente.
- **Grupo B (fine-tuning):** se carga preentrenada en ImageNet y se *descongela todo* para ajustar.
- **Grupo C (feature extraction):** se carga preentrenada y se *congela el backbone*; se entrena solo la capa final.


In [ ]:

def build_resnet18_scratch(num_classes: int) -> nn.Module:
    model = resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def build_resnet18_pretrained_finetune(num_classes: int) -> nn.Module:
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    # Fine-tuning: entrenar todo
    for p in model.parameters():
        p.requires_grad = True
    return model

def build_resnet18_pretrained_feature_extract(num_classes: int) -> nn.Module:
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    # Congelar backbone
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    # Entrenar solo la cabeza
    for p in model.fc.parameters():
        p.requires_grad = True
    return model

def count_trainable_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

A = build_resnet18_scratch(num_classes)
B = build_resnet18_pretrained_finetune(num_classes)
C = build_resnet18_pretrained_feature_extract(num_classes)

count_trainable_params(A), count_trainable_params(B), count_trainable_params(C)


## 4) Ejecutar experimentos
**Sugerencia:** corre primero Grupo C (más rápido), luego B, luego A (más costoso).

Notas de hiperparámetros:
- Grupo A suele requerir más épocas y/o LR cuidadoso.
- Grupo B puede usar LR menor para no “romper” features aprendidas.
- Grupo C puede usar LR un poco mayor porque solo entrena la cabeza.


In [ ]:

# Configuraciones (ajusta LR si lo necesitas)
cfg_A = TrainConfig(group_name="A_from_scratch", epochs=50, lr=3e-4, patience=7)
cfg_B = TrainConfig(group_name="B_finetune",    epochs=20, lr=1e-4, patience=5)
cfg_C = TrainConfig(group_name="C_feature_extract", epochs=10, lr=5e-4, patience=3)

results = {}

# Grupo C
model_C = build_resnet18_pretrained_feature_extract(num_classes)
results["C"] = train_one_experiment(model_C, train_loader, val_loader, cfg_C)

# Grupo B
model_B = build_resnet18_pretrained_finetune(num_classes)
results["B"] = train_one_experiment(model_B, train_loader, val_loader, cfg_B)

# Grupo A
model_A = build_resnet18_scratch(num_classes)
results["A"] = train_one_experiment(model_A, train_loader, val_loader, cfg_A)

# Evaluación en test
test_metrics = {}
for k in ["A","B","C"]:
    test_metrics[k] = test_model(results[k]["model"], test_loader)

test_metrics


## 5) Comparación de métricas
Construimos una tabla comparativa con:
- Accuracy final (test)
- Tiempo de entrenamiento (seg)
- Época de mejor validación
- Época de convergencia (estancamiento detectado)
- Peak GPU memory (MB)


In [ ]:

rows = []
for k, label in [("A","Grupo A (from scratch)"), ("B","Grupo B (fine-tuning)"), ("C","Grupo C (feature extraction)")]:
    r = results[k]
    rows.append({
        "Grupo": label,
        "Epochs (plan)": int(r["history"]["epoch"].max()),
        "Best Val Acc": float(r["best_val_acc"]),
        "Best Epoch (val)": int(r["best_epoch"]),
        "Convergence Epoch": int(r["convergence_epoch"]),
        "Test Acc (final)": float(test_metrics[k]["acc"]),
        "Train Time (sec)": float(r["train_time_sec"]),
        "Peak GPU Mem (MB)": float(r["peak_gpu_mem_mb"]),
        "Trainable Params": count_trainable_params(r["model"]),
    })

summary_df = pd.DataFrame(rows)
summary_df


## 6) Curvas de entrenamiento
Visualizamos **val accuracy** y **train/val loss** para cada grupo.


In [ ]:

def plot_metric(metric: str, title: str):
    plt.figure(figsize=(8,5))
    for k, label in [("A","A"), ("B","B"), ("C","C")]:
        h = results[k]["history"]
        plt.plot(h["epoch"], h[metric], label=label)
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_metric("val_acc", "Validation Accuracy por Grupo")
plot_metric("train_loss", "Train Loss por Grupo")
plot_metric("val_loss", "Validation Loss por Grupo")


## 7) Discusión: trade-offs y casos de uso
### 7.1 Resultados esperados (patrones típicos)
En problemas de visión como Flowers102, **transfer learning** suele:
- alcanzar **mejor accuracy** con **menos épocas**
- requerir **menos cómputo** (especialmente feature extraction)
- ser más robusto cuando el dataset no es enorme

### 7.2 Interpretación de métricas
- **Accuracy final (test):** calidad predictiva real.
- **Tiempo de entrenamiento:** costo operativo y tiempo-a-valor.
- **Épocas hasta convergencia:** eficiencia del aprendizaje; útil para planificar presupuestos de entrenamiento.
- **Memoria GPU:** determina viabilidad en infraestructura limitada.

### 7.3 Cuándo usar cada enfoque
**Grupo A (desde cero)**
- ✅ Útil si: tienes **dataset muy grande** y/o dominio muy distinto a ImageNet (p.ej. imágenes médicas con distribución distinta, imágenes satelitales) y quieres aprender features específicos.
- ✅ Útil si: necesitas **control total** del pipeline y no quieres depender de pesos externos.
- ❌ Costoso: más épocas, más riesgo de overfitting y más tuning.

**Grupo B (fine-tuning)**
- ✅ Mejor balance cuando: el dominio es similar/moderadamente distinto; quieres **máxima performance** con costo razonable.
- ✅ Permite adaptar features de alto nivel a las clases de flores.
- ⚠️ Requiere cuidado con LR (habitualmente menor) y regularización.

**Grupo C (feature extraction)**
- ✅ Ideal cuando: necesitas **rápido** y tienes infraestructura limitada.
- ✅ Muy útil como **baseline fuerte** o cuando el dataset es pequeño.
- ❌ Puede quedarse corto si se requiere adaptar significativamente el backbone.

### 7.4 Recomendaciones prácticas
- Si el objetivo es un MVP con buena calidad: empieza con **Grupo C**.
- Si buscas performance extra: pasa a **Grupo B**.
- Reserva **Grupo A** para casos con gran volumen de datos o dominios altamente especializados.


## 8) Extensiones sugeridas (opcional)
- Probar **ResNet50** (mejor capacidad, más costo).
- Agregar métricas: **Top-5 accuracy**, **matriz de confusión**, **F1 macro**.
- Usar `torch.compile()` (PyTorch 2.x) para acelerar.
- Probar *unfreezing gradual*: congelar al inicio y descongelar por etapas.
- Evaluar sensibilidad a `batch_size` y `img_size`.


## 9) Exportar resultados
Guardamos tabla resumen y curvas (si deseas) a disco.


In [ ]:

out_dir = "./outputs_flowers102"
os.makedirs(out_dir, exist_ok=True)

summary_path = os.path.join(out_dir, "summary_metrics.csv")
summary_df.to_csv(summary_path, index=False)

summary_path, summary_df
